In [ ]:
# 랜덤성 제어를 위한 표준 라이브러리 모듈을 불러옵니다.
import random
 # 고정 길이 큐를 쓰기 위해 deque를 가져옵니다.
from collections import deque
 # 파일/경로 처리를 위한 Path를 가져옵니다.
from pathlib import Path
 # 수치 계산용 NumPy를 불러옵니다.
import numpy as np
 # 표 형식 데이터 처리를 위한 pandas를 불러옵니다.
import pandas as pd
 # PyTorch 기본 패키지를 불러옵니다.
import torch
 # 신경망 레이어를 정의하기 위한 nn 모듈을 불러옵니다.
import torch.nn as nn
 # 함수형 활성화와 손실 계산에 사용할 F를 불러옵니다.
import torch.nn.functional as F
 # 최적화 알고리즘을 쓰기 위한 optim을 불러옵니다.
import torch.optim as optim
 # 결과 시각화를 위한 matplotlib를 불러옵니다.
import matplotlib.pyplot as plt
 # 히트맵 시각화를 위한 seaborn을 불러옵니다.
import seaborn as sns
 # 혼동행렬과 분류 리포트를 계산하기 위한 함수를 불러옵니다.
from sklearn.metrics import confusion_matrix, classification_report
 # 정규화 스케일링을 위한 MinMaxScaler를 불러옵니다.
from sklearn.preprocessing import MinMaxScaler
 # 클래스 불균형 보정을 위한 ADASYN을 불러옵니다.
from imblearn.over_sampling import ADASYN


dat = pd.read_csv(Path("../../data/processed/data_vif.csv"), index_col=0)

# 재현 가능한 실험을 위해 시드 값을 고정합니다.
SEED = 1
# 파이썬 random 모듈의 난수를 고정합니다.
random.seed(SEED)
# NumPy 난수를 고정합니다.
np.random.seed(SEED)
# PyTorch 난수를 고정합니다.
torch.manual_seed(SEED)

# GPU가 있으면 CUDA를, 없으면 CPU를 사용하도록 장치를 설정합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# DDQN에서 사용할 Q-Network 구조를 정의합니다.
# 데이터 수가 많지 않으므로 과적합 가능성을 염두에 두고 2개 은닉층을 사용합니다.
class QNetwork(nn.Module):
    # 입력 차원(state_dim)과 출력 차원(action_dim)을 받는 생성자입니다.
    def __init__(self, state_dim: int, action_dim: int):
        # 부모 클래스(nn.Module)의 초기화를 호출합니다.
        super().__init__()
        # 선형층과 ReLU를 순차적으로 쌓아 신경망 본체를 만듭니다.
        self.net = nn.Sequential(
            # 입력 특성을 96차원 은닉 표현으로 변환합니다.
            nn.Linear(state_dim, 96),
            # 비선형성을 부여합니다.
            nn.ReLU(),
            # 96차원을 64차원으로 줄입니다.
            nn.Linear(96, 64),
            # 다시 비선형성을 부여합니다.
            nn.ReLU(),
            # 마지막 출력층에서 행동 개수만큼 Q값을 출력합니다.
            nn.Linear(64, action_dim),
        )


    # 입력 텐서 x에 대해 Q값을 계산합니다.
    def forward(self, x):
        # 정의한 순차 모델을 그대로 통과시킵니다.
        return self.net(x)


# 경험을 저장하고 무작위 샘플링하기 위한 리플레이 버퍼를 정의합니다.
class ReplayBuffer:
    # 버퍼의 최대 용량을 받습니다.
    def __init__(self, capacity: int):
        # 최대 길이를 제한하는 deque를 사용합니다.
        self.buffer = deque(maxlen=capacity)

    # 하나의 전이(state, action, reward, next_state, done)를 버퍼에 저장합니다.
    def push(self, state, action, reward, next_state, done):
        # 전이 튜플을 추가합니다.
        self.buffer.append((state, action, reward, next_state, done))

    # 버퍼에서 미니배치를 무작위로 샘플링합니다.
    def sample(self, batch_size: int):
        # 배치 크기만큼 전이를 무작위로 뽑습니다.
        batch = random.sample(self.buffer, batch_size)
        # 각 항목을 states, actions, rewards, next_states, dones로 분리합니다.
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))

        # 학습에 사용할 텐서 형태로 변환해서 반환합니다.
        return (
            # 상태를 float 텐서로 바꾸고 현재 device로 옮깁니다.
            torch.FloatTensor(states).to(device),
            # 행동은 정수 인덱스이므로 LongTensor로 바꾸고 차원을 추가합니다.
            torch.LongTensor(actions).unsqueeze(1).to(device),
            # 보상은 float 텐서로 바꾸고 차원을 추가합니다.
            torch.FloatTensor(rewards).unsqueeze(1).to(device),
            # 다음 상태도 float 텐서로 바꾸고 device로 옮깁니다.
            torch.FloatTensor(next_states).to(device),
            # 종료 여부(done)는 float(0.0/1.0)으로 바꿔 마스크처럼 사용합니다.
            torch.FloatTensor(dones.astype(np.float32)).unsqueeze(1).to(device),
        )

    # 현재 저장된 전이 개수를 반환합니다.
    def __len__(self):
        return len(self.buffer)


# DDQN 에이전트를 정의합니다.
class DDQNAgent:
    # 상태 차원, 행동 차원, 학습 하이퍼파라미터를 받아 에이전트를 초기화합니다.
    def __init__(
        self,
        state_dim,
        action_dim,
        gamma=0.95,
        lr=5e-4,
        batch_size=64,
        memory_size=100_000,
        target_update=500,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay=0.995,
    ):
        # 행동 공간의 크기를 저장합니다.
        self.action_dim = action_dim
        # 할인율을 저장합니다.
        self.gamma = gamma
        # 미니배치 크기를 저장합니다.
        self.batch_size = batch_size
        # 타깃 네트워크를 얼마나 자주 동기화할지 저장합니다.
        self.target_update = target_update

        # 현재 epsilon 값을 저장합니다.
        self.epsilon = epsilon_start
        # epsilon의 하한을 저장합니다.
        self.epsilon_end = epsilon_end
        # epsilon 감소율을 저장합니다.
        self.epsilon_decay = epsilon_decay

        # 온라인 네트워크를 생성하고 device에 올립니다.
        self.online_net = QNetwork(state_dim, action_dim).to(device)
        # 타깃 네트워크를 생성하고 device에 올립니다.
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        # 시작 시 온라인 네트워크와 동일한 가중치로 맞춥니다.
        self.target_net.load_state_dict(self.online_net.state_dict())
        # 타깃 네트워크는 학습하지 않도록 eval 모드로 둡니다.
        self.target_net.eval()

        # 온라인 네트워크를 학습할 Adam 옵티마이저를 정의합니다.
        self.optimizer = optim.Adam(self.online_net.parameters(), lr=lr)
        # 경험 리플레이 버퍼를 생성합니다.
        self.memory = ReplayBuffer(memory_size)

        # 타깃 네트워크 갱신 횟수를 세는 카운터입니다.
        self.total_steps = 0

    # epsilon-greedy 방식으로 행동을 선택합니다.
    def act(self, state, epsilon: float = None):
        # 외부에서 epsilon을 넘기지 않으면 현재 epsilon을 사용합니다.
        if epsilon is None:
            epsilon = self.epsilon

        # 탐험 확률에 걸리면 무작위 행동을 선택합니다.
        if random.random() < epsilon:
            return random.randrange(self.action_dim)

        # 상태를 배치 차원 1개를 가진 텐서로 바꿉니다.
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        # 추론 시에는 기울기를 계산하지 않습니다.
        with torch.no_grad():
            # 온라인 네트워크로 Q값을 계산합니다.
            q_values = self.online_net(state_t)
        # 가장 큰 Q값을 주는 행동 인덱스를 반환합니다.
        return int(torch.argmax(q_values, dim=1).item())

    # epsilon 값을 점점 줄입니다.
    def decay_epsilon(self):
        # 하한 epsilon_end 아래로는 내려가지 않게 합니다.
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

    # DDQN 방식으로 네트워크를 1회 업데이트합니다.
    def update(self):
        # 버퍼에 충분한 샘플이 없으면 학습을 건너뜁니다.
        if len(self.memory) < self.batch_size:
            return None

        # 미니배치를 샘플링합니다.
        states, actions, rewards, next_states, dones = self.memory.sample(self.batch_size)

        # 현재 상태에 대한 Q(s,a)를 추출합니다.
        q_values = self.online_net(states).gather(1, actions)

        # 다음 상태에서 온라인 네트워크가 고른 행동을 구합니다.
        next_actions = self.online_net(next_states).argmax(dim=1, keepdim=True)
        # 타깃 네트워크에서 그 행동의 Q값을 평가합니다.
        next_q_values = self.target_net(next_states).gather(1, next_actions)

        # Bellman target을 계산합니다.
        target_q = rewards + (1 - dones) * self.gamma * next_q_values

        # 현재 Q와 타깃 Q 사이의 평균제곱오차를 손실로 사용합니다.
        loss = F.mse_loss(q_values, target_q.detach())

        # 기존 기울기를 초기화합니다.
        self.optimizer.zero_grad()
        # 역전파를 수행합니다.
        loss.backward()
        # 파라미터를 업데이트합니다.
        self.optimizer.step()

        # 스텝 수를 증가시킵니다.
        self.total_steps += 1
        # 일정 스텝마다 타깃 네트워크를 온라인 네트워크와 동기화합니다.
        if self.total_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.online_net.state_dict())

        # 손실값을 숫자로 반환합니다.
        return float(loss.item())


# ---- 데이터 준비 (Train/Valid/Test Split 먼저, 그 후 Scaling) ----

# 타깃 라벨 컬럼명을 지정합니다.
risk_label_col = "Risk_Label"

# 숫자형 컬럼만 특성으로 사용하되, 타깃 라벨은 제외합니다.
feature_cols = [c for c in dat.select_dtypes(include=[np.number]).columns if c != risk_label_col]
# 사용할 수 있는 숫자형 feature가 하나도 없으면 오류를 발생시킵니다.
if len(feature_cols) == 0:
    raise ValueError("학습에 사용할 수치형 feature가 없습니다.")

# 'Low Risk'를 0, 'High Risk'를 1로 바꾸고, 나머지는 0으로 처리합니다.
risk_label_data = dat[risk_label_col].map({'Low Risk': 0, 'High Risk': 1}).fillna(0).astype(int).to_numpy()
# feature 데이터를 결측치 0으로 채운 뒤 NumPy 배열로 변환합니다.
feature_data_raw = dat[feature_cols].fillna(0.0).to_numpy()
# 전체 시계열 길이를 저장합니다.
n_steps = len(dat)

# 데이터 분할 비율을 Train/Valid/Test = 50/30/20으로 설정합니다.
train_ratio, valid_ratio, test_ratio = 0.5, 0.3, 0.2
# 학습 구간의 길이를 계산합니다.
n_train = int(n_steps * train_ratio)
# 검증 구간의 길이를 계산합니다.
n_valid = int(n_steps * valid_ratio)

# 학습 인덱스 구간을 만듭니다.
train_indices = np.arange(0, n_train)
# 검증 인덱스 구간을 만듭니다.
valid_indices = np.arange(n_train, n_train + n_valid)
# 테스트 인덱스 구간을 만듭니다.
test_indices = np.arange(n_train + n_valid, n_steps)

# 학습용 원본 feature와 label을 분리합니다.
train_feature_raw = feature_data_raw[train_indices]
# 학습 라벨을 분리합니다.
train_label = risk_label_data[train_indices]

# 검증용 원본 feature와 label을 분리합니다.
valid_feature_raw = feature_data_raw[valid_indices]
# 검증 라벨을 분리합니다.
valid_label = risk_label_data[valid_indices]
# 검증 세트가 너무 짧으면 튜닝과 선택용으로 나눌 수 없으므로 오류를 발생시킵니다.
if len(valid_label) < 2:
    raise ValueError("Validation set 길이가 너무 짧아 tuning/selection 분할이 불가능합니다. split 비율을 조정하세요.")
# 검증 세트를 튜닝용과 선택용으로 반반 나눕니다.
valid_split_point = len(valid_label) // 2
# 앞 절반은 하이퍼파라미터 튜닝에 사용합니다.
valid_tune_feature_raw = valid_feature_raw[:valid_split_point]
# 튜닝용 라벨입니다.
valid_tune_label = valid_label[:valid_split_point]
# 뒤 절반은 최종 선택 및 평가에 사용합니다.
valid_select_feature_raw = valid_feature_raw[valid_split_point:]
# 선택용 라벨입니다.
valid_select_label = valid_label[valid_split_point:]

# 테스트용 원본 feature를 분리합니다.
test_feature_raw = feature_data_raw[test_indices]
# 테스트 라벨을 분리합니다.
test_label = risk_label_data[test_indices]

# 각 데이터 구간의 클래스 비율을 출력하는 함수입니다.
def summarize_class_ratio(name, y):
    # 전체 샘플 수를 계산합니다.
    n_total = len(y)
    # 클래스 0 개수를 계산합니다.
    n0 = int((y == 0).sum())
    # 클래스 1 개수를 계산합니다.
    n1 = int((y == 1).sum())
    # 클래스 0 비율을 계산합니다.
    r0 = n0 / n_total if n_total > 0 else 0.0
    # 클래스 1 비율을 계산합니다.
    r1 = n1 / n_total if n_total > 0 else 0.0
    # 클래스 1 대비 클래스 0 비율을 계산합니다.
    imbalance = (n1 / max(n0, 1)) if n_total > 0 else 0.0
    # 보기 좋게 요약 정보를 출력합니다.
    print(f"{name:>5} | n={n_total:4d} | class0={n0:4d} ({r0:.2%}) | class1={n1:4d} ({r1:.2%}) | c1/c0={imbalance:.4f})")
    # 어느 한 클래스라도 없으면 학습이나 평가가 불안정하므로 오류를 발생시킵니다.
    if n0 == 0 or n1 == 0:
        raise ValueError(f"{name} set에 클래스 0 또는 1이 없습니다. split 비율 또는 기간을 조정하세요.")

# ADASYN 적용 전 클래스 비율을 확인합니다.
print("Class Ratio Summary (Before ADASYN)")
# 학습 세트 비율을 출력합니다.
summarize_class_ratio("Train", train_label)
# 튜닝용 검증 세트 비율을 출력합니다.
summarize_class_ratio("ValTun", valid_tune_label)
# 선택용 검증 세트 비율을 출력합니다.
summarize_class_ratio("ValSel", valid_select_label)
# 테스트 세트 비율을 출력합니다.
summarize_class_ratio("Test", test_label)

# 보상 계산에 사용할 class_ratio는 ADASYN 적용 전 원본 train 기준으로 고정합니다.
n_negative_original = (train_label == 0).sum()
# 원본 train의 양성 클래스 수를 계산합니다.
n_positive_original = (train_label == 1).sum()
# 보상에 사용할 양성/음성 비율을 계산합니다.
class_ratio_reward = n_positive_original / max(n_negative_original, 1)
# 계산된 비율을 출력합니다.
print(f"Reward class ratio (original train): {class_ratio_reward:.4f}")

# ---- Train Set에 ADASYN 적용 ----

# ADASYN 객체를 생성합니다.
adasyn = ADASYN(random_state=SEED, n_neighbors=5)
# 학습 데이터에만 ADASYN을 적용해 샘플 수를 늘립니다.
train_feature_adasyn, train_label_adasyn = adasyn.fit_resample(train_feature_raw, train_label)

# ADASYN 이후 클래스 비율을 확인합니다.
print("\nClass Ratio Summary (After ADASYN)")
# ADASYN 적용 후 학습 비율을 출력합니다.
summarize_class_ratio("Train", train_label_adasyn)

# ADASYN으로 보강된 데이터를 학습용 데이터로 다시 사용합니다.
train_feature_raw = train_feature_adasyn
# ADASYN으로 보강된 라벨을 학습 라벨로 다시 사용합니다.
train_label = train_label_adasyn

# Min-Max 스케일러를 생성합니다.
scaler = MinMaxScaler()
# 스케일러는 학습 데이터에만 적합시킵니다.
scaler.fit(train_feature_raw)

# 학습 특성을 0~1 범위로 정규화합니다.
train_feature = scaler.transform(train_feature_raw).astype(np.float32)
# 튜닝용 검증 특성을 정규화합니다.
valid_tune_feature = scaler.transform(valid_tune_feature_raw).astype(np.float32)
# 선택용 검증 특성을 정규화합니다.
valid_select_feature = scaler.transform(valid_select_feature_raw).astype(np.float32)
# 테스트 특성을 정규화합니다.
test_feature = scaler.transform(test_feature_raw).astype(np.float32)

# 전체 분할 정보를 출력합니다.
print("=" * 60)
# 분할 정보 섹션 제목입니다.
print("Data Split Information")
# 구분선을 출력합니다.
print("=" * 60)
# 전체 샘플 수를 출력합니다.
print(f"Total samples: {n_steps}")
# ADASYN 이후 학습 세트 크기를 출력합니다.
print(f"Train set (after ADASYN): {len(train_label)} samples")
# 학습 세트의 클래스별 개수를 출력합니다.
print(f"  - Class 0: {(train_label==0).sum()}, Class 1: {(train_label==1).sum()}")
# 튜닝용 검증 세트 크기를 출력합니다.
print(f"Valid(Tune) set: {len(valid_tune_label)} samples")
# 튜닝용 검증 세트의 클래스별 개수를 출력합니다.
print(f"  - Class 0: {(valid_tune_label==0).sum()}, Class 1: {(valid_tune_label==1).sum()}")
# 선택용 검증 세트 크기를 출력합니다.
print(f"Valid(Select) set: {len(valid_select_label)} samples")
# 선택용 검증 세트의 클래스별 개수를 출력합니다.
print(f"  - Class 0: {(valid_select_label==0).sum()}, Class 1: {(valid_select_label==1).sum()}")
# 테스트 세트 크기를 출력합니다.
print(f"Test set: {len(test_indices)} samples ({test_ratio*100:.1f}%)")
# 테스트 세트의 클래스별 개수를 출력합니다.
print(f"  - Class 0: {(test_label==0).sum()}, Class 1: {(test_label==1).sum()}")

# ADASYN 이후 학습 세트의 클래스 비율을 다시 계산합니다.
n_negative = (train_label == 0).sum()
# ADASYN 이후 양성 클래스 수를 다시 계산합니다.
n_positive = (train_label == 1).sum()
# ADASYN 이후 양성/음성 비율을 계산합니다.
class_ratio_after_adasyn = n_positive / max(n_negative, 1)
# 보상에 사용할 class_ratio를 원본 기준으로 고정합니다.
class_ratio = class_ratio_reward

# ADASYN 이후의 클래스 비율과 보상 비율을 출력합니다.
print(f"\nClass ratio (train set after ADASYN): {class_ratio_after_adasyn:.4f}")
# 보상에 쓰는 비율이 원본 train 기준임을 출력합니다.
print(f"Class ratio used for reward (fixed original train): {class_ratio:.4f}")
# 구분선을 출력합니다.
print("=" * 60)

# 상태 차원은 feature 개수와 같습니다.
state_dim = len(feature_cols)
# 행동은 두 가지(0, 1)입니다.
action_dim = 2


# 보상 함수를 정의합니다.
def compute_reward(action, actual_label, class_ratio):
    # 행동과 라벨을 정수형으로 정규화합니다.
    action = int(action)
    # 라벨도 정수형으로 정규화합니다.
    actual_label = int(actual_label)

    # 실제 라벨이 0인 경우를 처리합니다.
    if actual_label == 0:
        # 음성 클래스를 음성으로 맞히면 class_ratio만큼 보상을 줍니다.
        if action == 0:
            return class_ratio
        # 음성 클래스를 양성으로 잘못 분류하면 같은 크기의 벌점을 줍니다.
        else:
            return -class_ratio
    # 실제 라벨이 1인 경우를 처리합니다.
    else:
        # 양성 클래스를 양성으로 맞히면 1.0 보상을 줍니다.
        if action == 1:
            return 1.0
        # 양성 클래스를 음성으로 잘못 분류하면 -1.0 벌점을 줍니다.
        else:
            return -1.0


# ---- Validation 기반 하이퍼파라미터 Grid Search ----
from itertools import product

# 탐색할 하이퍼파라미터 조합을 정의합니다.
param_grid = {
    # 할인율 후보입니다.
    'gamma': [0.9, 0.99],
    # 학습률 후보입니다.
    'lr': [1e-4, 1e-3],
    # epsilon 감소율 후보입니다.
    'epsilon_decay': [0.99, 0.999],
    # 미니배치 크기 후보입니다.
    'batch_size': [32, 64],
}

# G-mean, 민감도, 특이도를 계산하는 함수입니다.
def compute_gmean(y_true, y_pred):
    # 혼동행렬을 계산합니다.
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # tn, fp, fn, tp를 꺼냅니다.
    tn, fp, fn, tp = cm.ravel()
    # 양성 클래스 재현율(민감도)을 계산합니다.
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    # 음성 클래스 재현율(특이도)을 계산합니다.
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    # G-mean을 계산해 반환합니다.
    gmean = float(np.sqrt(sensitivity * specificity))
    return gmean, sensitivity, specificity

# 네트워크 출력에서 threshold 기준으로 예측 라벨을 만듭니다.
def predict_labels_with_threshold(model, features, threshold=0.5):
    # 예측 결과를 저장할 리스트입니다.
    preds = []
    # 추론 시 기울기를 계산하지 않습니다.
    with torch.no_grad():
        # 각 샘플을 순회합니다.
        for t in range(len(features)):
            # 현재 상태를 배치 차원으로 감싸 텐서로 만듭니다.
            state_t = torch.FloatTensor(features[t]).unsqueeze(0).to(device)
            # 모델에서 Q값을 구합니다.
            q_values = model(state_t)
            # softmax로 양성 클래스 확률을 근사합니다.
            prob_pos = torch.softmax(q_values, dim=1)[0, 1].item()
            # threshold 이상이면 1, 아니면 0으로 예측합니다.
            preds.append(1 if prob_pos >= threshold else 0)
    # NumPy 배열로 반환합니다.
    return np.array(preds)

# valid_tune에서 G-mean이 가장 좋은 threshold를 찾습니다.
def tune_threshold_by_gmean(model, x_valid, y_valid, threshold_grid=None):
    # threshold 후보가 없으면 0.1~0.9 구간을 촘촘히 탐색합니다.
    if threshold_grid is None:
        threshold_grid = np.linspace(0.1, 0.9, 81)

    # 현재까지의 최적 threshold를 저장합니다.
    best_thr = 0.5
    # 현재까지의 최고 G-mean을 저장합니다.
    best_gmean = -1.0

    # 각 threshold를 평가합니다.
    for thr in threshold_grid:
        # threshold를 적용해 예측을 생성합니다.
        preds = predict_labels_with_threshold(model, x_valid, threshold=float(thr))
        # G-mean을 계산합니다.
        gmean, _, _ = compute_gmean(y_valid, preds)
        # 더 좋은 threshold면 갱신합니다.
        if gmean > best_gmean:
            best_gmean = gmean
            best_thr = float(thr)

    # 최적 threshold와 그때의 G-mean을 반환합니다.
    return best_thr, best_gmean

# 고정 threshold로 검증 성능을 계산합니다.
def evaluate_with_fixed_threshold(model, x_valid, y_valid, fixed_threshold):
    # threshold를 적용해 예측을 만듭니다.
    preds = predict_labels_with_threshold(model, x_valid, threshold=float(fixed_threshold))
    # G-mean, 민감도, 특이도를 계산합니다.
    gmean, sensitivity, specificity = compute_gmean(y_valid, preds)
    return gmean, sensitivity, specificity

# 각 하이퍼파라미터 조합에 대한 결과를 저장할 리스트입니다.
grid_search_results = []
# 모든 조합을 미리 생성합니다.
param_combinations = list(product(*param_grid.values()))
# 파라미터 이름 목록을 저장합니다.
param_names = list(param_grid.keys())

# 그리드 서치 시작 안내를 출력합니다.
print("\n" + "=" * 80)
# 전체 조합 개수를 출력합니다.
print(f"HYPERPARAMETER GRID SEARCH (Total: {len(param_combinations)} combinations)")
# 구분선을 출력합니다.
print("=" * 80)

# 조기 종료 기준입니다.
patience = 25 # Increased from 15
# 최소 학습 에피소드 수입니다.
min_episodes = 60 # Increased from 40
# 선택용 검증 G-mean의 현재 최고값입니다.
best_overall_select_gmean = 0.0
# 최적 하이퍼파라미터를 저장할 변수입니다.
best_overall_params = None


# 각 하이퍼파라미터 조합을 순회하며 학습합니다.
for idx, params in enumerate(param_combinations):
    # 현재 조합을 딕셔너리로 만듭니다.
    param_dict = dict(zip(param_names, params))
    # 각 파라미터를 꺼냅니다.
    gamma, lr, epsilon_decay, batch_size = params

    # 현재 조합을 출력합니다.
    print(f"\n[{idx+1}/{len(param_combinations)}] γ={gamma:.2f}, lr={lr:.0e}, ε={epsilon_decay:.3f}, bs={batch_size}")

    # 현재 조합으로 튜닝용 에이전트를 생성합니다.
    agent_tune = DDQNAgent(
        # 상태 차원을 전달합니다.
        state_dim=state_dim,
        # 행동 차원을 전달합니다.
        action_dim=action_dim,
        # 할인율을 전달합니다.
        gamma=gamma,
        # 학습률을 전달합니다.
        lr=lr,
        # 배치 크기를 전달합니다.
        batch_size=batch_size,
        # 리플레이 버퍼 크기를 줄여 튜닝 속도를 높입니다.
        memory_size=50_000,
        # 타깃 네트워크 갱신 주기를 설정합니다.
        target_update=200,
        # 시작 epsilon을 1.0으로 둡니다.
        epsilon_start=1.0,
        # epsilon 하한을 설정합니다.
        epsilon_end=0.05,
        # epsilon 감소율을 전달합니다.
        epsilon_decay=epsilon_decay,
    )

    # 튜닝 학습 에피소드 수입니다.
    episodes_tune = 150 # Increased from 100
# 한 에피소드에서 진행할 최대 스텝 수입니다.
    max_steps_tune = 500
# 현재 조합의 최고 튜닝 G-mean입니다.
    best_tune_gmean = 0.0
# early stopping용 카운터입니다.
    patience_counter = 0

    # 지정된 에피소드 수 동안 학습합니다.
    for ep in range(1, episodes_tune + 1):
        # 길이 제한을 고려해 시작 인덱스를 무작위로 뽑습니다.
        start_idx = np.random.randint(0, len(train_feature) - max_steps_tune - 1)
        # 에피소드 누적 보상입니다.
        episode_reward = 0.0

        # 연속 구간을 따라 환경 상호작용을 시뮬레이션합니다.
        for i in range(max_steps_tune):
            # 현재 상태를 꺼냅니다.
            state = train_feature[start_idx + i]
            # 다음 상태를 꺼냅니다.
            next_state = train_feature[start_idx + i + 1] if start_idx + i + 1 < len(train_feature) else train_feature[start_idx + i]
            # 현재 시점의 실제 라벨을 가져옵니다.
            actual_label = train_label[start_idx + i]

            # 현재 상태에서 행동을 선택합니다.
            action = agent_tune.act(state)
            # 보상 함수를 통해 보상을 계산합니다.
            reward = compute_reward(action, actual_label, class_ratio)
            # 마지막 스텝인지 여부를 표시합니다.
            done = i == (max_steps_tune - 1)

            # 전이를 리플레이 버퍼에 저장합니다.
            agent_tune.memory.push(state, action, reward, next_state, done)
            # 네트워크를 한 번 업데이트합니다.
            _ = agent_tune.update()
            # 누적 보상에 더합니다.
            episode_reward += reward

        # epsilon을 감소시킵니다.
        agent_tune.decay_epsilon()

        # 5 에피소드마다 검증 성능을 확인합니다.
        if ep % 5 == 0:
            # 검증 시에는 평가 모드로 전환합니다.
            agent_tune.online_net.eval()
            # 검증 예측을 저장할 리스트입니다.
            valid_pred = []
            # 기울기 계산 없이 추론합니다.
            with torch.no_grad():
                # 튜닝용 검증 세트를 순회합니다.
                for t in range(len(valid_tune_label)):
                    # 현재 검증 상태를 가져옵니다.
                    state = valid_tune_feature[t]
                    # 텐서로 바꿉니다.
                    state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                    # Q값을 계산합니다.
                    q_values = agent_tune.online_net(state_t)
                    # 가장 큰 Q값의 행동을 선택합니다.
                    action = int(torch.argmax(q_values, dim=1).item())
                    # 예측을 저장합니다.
                    valid_pred.append(action)

            # 리스트를 NumPy 배열로 변환합니다.
            valid_pred = np.array(valid_pred)
            # 튜닝용 검증 G-mean을 계산합니다.
            tune_gmean, tune_sensitivity, tune_specificity = compute_gmean(valid_tune_label, valid_pred)
            # 다시 학습 모드로 돌립니다.
            agent_tune.online_net.train()

            # 현재 G-mean이 최고면 기록을 갱신합니다.
            if tune_gmean > best_tune_gmean:
                best_tune_gmean = tune_gmean
                patience_counter = 0
            # 아니면 patience를 1 늘립니다.
            else:
                patience_counter += 1

            # 20 에피소드마다 로그를 출력합니다.
            if ep % 20 == 0:
                print(f"  Ep {ep:3d} | Tune G-mean: {tune_gmean:.4f} (Sens: {tune_sensitivity:.4f}, Spec: {tune_specificity:.4f}) | Best: {best_tune_gmean:.4f}")

            # 충분히 학습했고 개선이 없으면 조기 종료합니다.
            if ep > min_episodes and patience_counter >= patience:
                break

    # 조합 학습이 끝나면 평가 모드로 전환합니다.
    agent_tune.online_net.eval()
    # 선택용 검증 예측을 저장합니다.
    valid_select_pred = []
    # 기울기 계산 없이 추론합니다.
    with torch.no_grad():
        # 선택용 검증 세트를 순회합니다.
        for t in range(len(valid_select_label)):
            # 현재 상태를 꺼냅니다.
            state = valid_select_feature[t]
            # 배치 차원으로 감싼 텐서로 만듭니다.
            state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            # Q값을 계산합니다.
            q_values = agent_tune.online_net(state_t)
            # 가장 큰 Q값의 행동을 선택합니다.
            action = int(torch.argmax(q_values, dim=1).item())
            # 예측을 리스트에 추가합니다.
            valid_select_pred.append(action)
    # NumPy 배열로 바꿉니다.
    valid_select_pred = np.array(valid_select_pred)
    # 선택용 검증 G-mean을 계산합니다.
    select_gmean, select_sensitivity, select_specificity = compute_gmean(valid_select_label, valid_select_pred)


    # 현재 조합의 결과를 저장합니다.
    grid_search_results.append({
        # 하이퍼파라미터 자체를 저장합니다.
        'params': param_dict,
        # 튜닝 세트에서 얻은 최고 G-mean을 저장합니다.
        'best_tune_gmean': best_tune_gmean,
        # 선택 세트 G-mean을 저장합니다.
        'select_gmean': select_gmean,
        # 선택 세트 민감도를 저장합니다.
        'select_sensitivity': select_sensitivity,
        # 선택 세트 특이도를 저장합니다.
        'select_specificity': select_specificity,
    })

    # 지금까지의 최고 선택 G-mean보다 좋으면 갱신합니다.
    if select_gmean > best_overall_select_gmean:
        best_overall_select_gmean = select_gmean
        best_overall_params = param_dict

# 선택 G-mean 기준으로 결과를 정렬합니다.
grid_search_results.sort(key=lambda x: x['select_gmean'], reverse=True)


# 상위 5개 하이퍼파라미터를 출력합니다.
print("\n" + "=" * 80)
# 상위 결과 섹션 제목입니다.
print("TOP 5 HYPERPARAMETERS")
# 구분선을 출력합니다.
print("=" * 80)

# 상위 5개를 순서대로 출력합니다.
for i, result in enumerate(grid_search_results[:5]):
    print(f"{i+1}. Select G-mean: {result['select_gmean']:.4f} | Tune Best: {result['best_tune_gmean']:.4f} | Params: {result['params']}")

# 최적 하이퍼파라미터 섹션을 출력합니다.
print("\n" + "=" * 80)
# 섹션 제목입니다.
print("BEST HYPERPARAMETERS")
# 구분선을 출력합니다.
print("=" * 80)
# 최적 파라미터를 한 항목씩 출력합니다.
for k, v in best_overall_params.items():
    print(f"{k}: {v}")
# 최적 선택 G-mean을 출력합니다.
print(f"Best Selection G-mean: {best_overall_select_gmean:.4f}")
# 구분선을 출력합니다.
print("=" * 80)


# ---- 최적 파라미터로 재학습 ----
# 최적 하이퍼파라미터로 전체 학습을 다시 수행합니다.
print("\nTraining with best hyperparameters...")
# 최적 파라미터로 최종 에이전트를 생성합니다.
agent = DDQNAgent(
    # 상태 차원을 넣습니다.
    state_dim=state_dim,
    # 행동 차원을 넣습니다.
    action_dim=action_dim,
    # 최적 gamma를 넣습니다.
    gamma=best_overall_params['gamma'],
    # 최적 학습률을 넣습니다.
    lr=best_overall_params['lr'],
    # 최적 배치 크기를 넣습니다.
    batch_size=best_overall_params['batch_size'],
    # 최종 학습도 메모리 크기는 제한합니다.
    memory_size=50_000,
    # 타깃 네트워크 갱신 주기를 넣습니다.
    target_update=200,
    # 시작 epsilon을 넣습니다.
    epsilon_start=1.0,
    # epsilon 하한을 넣습니다.
    epsilon_end=0.05,
    # 최적 epsilon 감소율을 넣습니다.
    epsilon_decay=best_overall_params['epsilon_decay'],
)


# 최종 학습 에피소드 수입니다.
episodes = 600 # Increased from 250
# 한 에피소드당 최대 스텝 수입니다.
max_steps = 500
# 에피소드별 보상을 저장합니다.
scores = []
# 검증 점수를 저장합니다.
valid_scores = []
# 가장 좋은 검증 G-mean을 저장합니다.
best_ckpt_valid_gmean = -1.0
# 가장 좋은 체크포인트의 threshold를 저장합니다.
best_ckpt_threshold = 0.5
# 가장 좋은 체크포인트의 epoch을 저장합니다.
best_ckpt_epoch = 0
# 가장 좋은 체크포인트의 state_dict를 저장합니다.
best_ckpt_state_dict = None

# valid_tune에서 threshold를 한 번만 탐색하고 이후에는 고정합니다.
fixed_eval_threshold, fixed_eval_tune_gmean = tune_threshold_by_gmean(
    # 현재 온라인 네트워크를 사용합니다.
    agent.online_net, valid_tune_feature, valid_tune_label
)
# 고정 threshold 정보를 출력합니다.
print(f"Fixed validation threshold: {fixed_eval_threshold:.3f} (tuned once on valid_tune, G-mean: {fixed_eval_tune_gmean:.4f})")

# 최종 학습 루프를 시작합니다.
for ep in range(1, episodes + 1):
    # 현재 에피소드 누적 보상입니다.
    episode_reward = 0.0
    # 긴 구간을 무작위로 샘플링합니다.
    start_idx = np.random.randint(0, len(train_feature) - max_steps - 1)

    # 한 에피소드 동안 step 단위로 학습합니다.
    for i in range(max_steps):
        # 현재 상태를 읽습니다.
        state = train_feature[start_idx + i]
        # 다음 상태를 읽습니다.
        next_state = train_feature[start_idx + i + 1] if start_idx + i + 1 < len(train_feature) else train_feature[start_idx + i]
        # 현재 시점 라벨을 읽습니다.
        actual_label = train_label[start_idx + i]

        # 현재 정책으로 행동을 선택합니다.
        action = agent.act(state)
        # 보상을 계산합니다.
        reward = compute_reward(action, actual_label, class_ratio)
        # 마지막 스텝 여부입니다.
        done = i == (max_steps - 1)

        # 전이를 저장합니다.
        agent.memory.push(state, action, reward, next_state, done)
        # 모델을 한 번 업데이트합니다.
        _ = agent.update()
        # 에피소드 보상을 누적합니다.
        episode_reward += reward

    # 에피소드가 끝나면 epsilon을 줄입니다.
    agent.decay_epsilon()
# 이번 에피소드의 보상을 기록합니다.
    scores.append(episode_reward)

    # 10 에피소드마다 검증 성능을 계산합니다.
    if ep % 10 == 0:
        # 검증 시 평가 모드로 전환합니다.
        agent.online_net.eval()
        # 고정 threshold로 선택용 검증 성능을 계산합니다.
        valid_gmean, valid_sensitivity, valid_specificity = evaluate_with_fixed_threshold(
            # 최종 모델, 선택용 검증 데이터, 고정 threshold를 전달합니다.
            agent.online_net, valid_select_feature, valid_select_label, fixed_eval_threshold
        )
# 검증 점수를 저장합니다.
        valid_scores.append(valid_gmean)

        # 현재 검증 G-mean이 최고면 체크포인트를 갱신합니다.
        if valid_gmean > best_ckpt_valid_gmean:
            best_ckpt_valid_gmean = valid_gmean
            best_ckpt_threshold = fixed_eval_threshold
            best_ckpt_epoch = ep
            best_ckpt_state_dict = {k: v.detach().cpu().clone() for k, v in agent.online_net.state_dict().items()}
        # 다시 학습 모드로 돌립니다.
        agent.online_net.train()

        # 학습 로그를 출력합니다.
        print(f"Episode {ep:3d} | Train Reward: {np.mean(scores[-10:]):8.5f} | Valid G-mean: {valid_gmean:.4f} (Sens: {valid_sensitivity:.4f}, Spec: {valid_specificity:.4f}) | Fixed Thr: {fixed_eval_threshold:.3f} | Best G: {best_ckpt_valid_gmean:.4f}")

# 전체 학습이 끝났음을 출력합니다.
print("최종 학습 완료")
# 저장된 최고 체크포인트가 있으면 복원합니다.
if best_ckpt_state_dict is not None:
    # 온라인 네트워크에 최고 가중치를 복원합니다.
    agent.online_net.load_state_dict(best_ckpt_state_dict)
    # 타깃 네트워크도 동일하게 맞춥니다.
    agent.target_net.load_state_dict(best_ckpt_state_dict)
    # 복원된 체크포인트 정보를 출력합니다.
    print(f"Best checkpoint restored: epoch={best_ckpt_epoch}, valid_gmean={best_ckpt_valid_gmean:.4f}, threshold={best_ckpt_threshold:.3f}")


# ---- Test Set에서 최종 성능 평가 ----

# 평가를 위해 온라인 네트워크를 eval 모드로 전환합니다.
agent.online_net.eval()

# 학습 중 선택된 best checkpoint의 threshold를 사용합니다.
best_threshold = best_ckpt_threshold
# 해당 threshold에서의 검증 G-mean을 함께 저장합니다.
best_threshold_gmean = best_ckpt_valid_gmean
# 선택된 threshold를 출력합니다.
print(f"Best threshold from checkpoint: {best_threshold:.3f} (Valid G-mean: {best_threshold_gmean:.4f}, Epoch: {best_ckpt_epoch})")

# 테스트 세트에 대한 예측을 생성합니다.
test_predictions = predict_labels_with_threshold(
    # 최종 모델과 테스트 특성을 전달합니다.
    agent.online_net, test_feature, threshold=best_threshold
)

# 테스트 세트 혼동행렬을 계산합니다.
cm_test = confusion_matrix(test_label, test_predictions, labels=[0, 1])

# 혼동행렬에서 개별 값을 꺼냅니다.
tn, fp, fn, tp = cm_test.ravel()
# 테스트 민감도를 계산합니다.
sensitivity_test = tp / (tp + fn) if (tp + fn) > 0 else 0
# 테스트 특이도를 계산합니다.
specificity_test = tn / (tn + fp) if (tn + fp) > 0 else 0
# 테스트 G-mean을 계산합니다.
gmean_test = np.sqrt(sensitivity_test * specificity_test)

# 테스트 결과 섹션 제목을 출력합니다.
print("\n" + "=" * 60)
# 섹션 제목입니다.
print("TEST SET - DDQN 분류 모델 최종 성능 평가")
# 구분선을 출력합니다.
print("=" * 60)
# G-mean을 출력합니다.
print(f"\nG-mean: {gmean_test:.4f}")
# 혼동행렬 제목을 출력합니다.
print(f"\nConfusion Matrix:")
# 혼동행렬을 출력합니다.
print(cm_test)
# TN과 FP를 출력합니다.
print(f"\nTN: {cm_test[0, 0]}, FP: {cm_test[0, 1]}")
# FN과 TP를 출력합니다.
print(f"FN: {cm_test[1, 0]}, TP: {cm_test[1, 1]}")

# 분류 리포트를 출력합니다.
print(f"\n분류 리포트:")
# 클래스 이름을 지정해 리포트를 출력합니다.
print(classification_report(test_label, test_predictions,
                          labels=[0, 1],
                          target_names=['Class 0 (Negative)', 'Class 1 (Positive)'],
                          zero_division=0))

# 테스트 세트 혼동행렬을 시각화합니다.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽 축에 개수 기반 혼동행렬을 그립니다.
ax1 = axes[0]
# 혼동행렬 개수 버전 히트맵을 그립니다.
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1,
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['Actual 0', 'Actual 1'])
# 그래프 제목을 설정합니다.
ax1.set_title('Test Set - Confusion Matrix (Count)', fontsize=12, fontweight='bold')
# y축 라벨을 설정합니다.
ax1.set_ylabel('True Label')
# x축 라벨을 설정합니다.
ax1.set_xlabel('Predicted Label')

# 행 합으로 나눈 퍼센트 혼동행렬을 계산합니다.
cm_test_row_sums = cm_test.sum(axis=1, keepdims=True)
# 0으로 나누지 않도록 안전하게 퍼센트값을 계산합니다.
cm_test_percent = np.divide(cm_test.astype(float), cm_test_row_sums, out=np.zeros_like(cm_test, dtype=float), where=cm_test_row_sums != 0) * 100
# 오른쪽 축을 가져옵니다.
ax2 = axes[1]
# 혼동행렬 퍼센트 버전 히트맵을 그립니다.
sns.heatmap(cm_test_percent, annot=True, fmt='.2f', cmap='Blues', cbar=True, ax=ax2,
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['Actual 0', 'Actual 1'],
            cbar_kws={'label': 'Percentage (%)'})
# 그래프 제목을 설정합니다.
ax2.set_title('Test Set - Confusion Matrix (Percentage)', fontsize=12, fontweight='bold')
# y축 라벨을 설정합니다.
ax2.set_ylabel('True Label')
# x축 라벨을 설정합니다.
ax2.set_xlabel('Predicted Label')

# 레이아웃을 정리합니다.
plt.tight_layout()
# 그래프를 보여줍니다.
plt.show()

# 정밀도와 F1을 직접 계산합니다.
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
# F1-score를 계산합니다.
f1_score = 2 * (precision * sensitivity_test) / (precision + sensitivity_test) if (precision + sensitivity_test) > 0 else 0

# 상세 지표 섹션 제목을 출력합니다.
print(f"\n상세 메트릭 (Test Set):")
# 민감도를 출력합니다.
print(f"Sensitivity (Recall for Class 1): {sensitivity_test:.4f}")
# 특이도를 출력합니다.
print(f"Specificity (Recall for Class 0): {specificity_test:.4f}")
# 정밀도를 출력합니다.
print(f"Precision (Class 1): {precision:.4f}")
# F1-score를 출력합니다.
print(f"F1-Score: {f1_score:.4f}")
# 구분선을 출력합니다.
print("=" * 60)

# 검증 세트와 테스트 세트를 비교합니다.
print("\n" + "=" * 60)
# 비교 섹션 제목을 출력합니다.
print("VALIDATION SET vs TEST SET 성능 비교")
# 구분선을 출력합니다.
print("=" * 60)

# 선택용 검증 세트를 다시 평가합니다.
agent.online_net.eval()
# 고정 threshold로 검증 예측을 만듭니다.
valid_predictions = predict_labels_with_threshold(
    # 최종 모델과 선택용 검증 데이터를 전달합니다.
    agent.online_net, valid_select_feature, threshold=best_threshold
)
# 검증 혼동행렬을 계산합니다.
cm_valid = confusion_matrix(valid_select_label, valid_predictions, labels=[0, 1])
# 검증 혼동행렬의 원소를 꺼냅니다.
tn_v, fp_v, fn_v, tp_v = cm_valid.ravel()
# 검증 민감도를 계산합니다.
sensitivity_valid = tp_v / (tp_v + fn_v) if (tp_v + fn_v) > 0 else 0
# 검증 특이도를 계산합니다.
specificity_valid = tn_v / (tn_v + fp_v) if (tn_v + fp_v) > 0 else 0 # Corrected: should be based on its own fp_v
# 검증 G-mean을 계산합니다.
gmean_valid = np.sqrt(sensitivity_valid * specificity_valid)

# 선택용 검증 G-mean을 출력합니다.
print(f"\nValidation(Selection) Set - G-mean: {gmean_valid:.4f}")
# 테스트 G-mean을 출력합니다.
print(f"Test Set - G-mean: {gmean_test:.4f}")
# 두 값의 차이를 출력합니다.
print(f"Difference (Valid - Test): {gmean_valid - gmean_test:.4f}")
# 검증 혼동행렬 제목을 출력합니다.
print("\nValidation Confusion Matrix:")
# 검증 혼동행렬을 출력합니다.
print(cm_valid)
# 테스트 혼동행렬 제목을 출력합니다.
print("\nTest Confusion Matrix:")
# 테스트 혼동행렬을 출력합니다.
print(cm_test)
# 구분선을 출력합니다.
print("=" * 60)


Class Ratio Summary (Before ADASYN)
Train | n=2054 | class0=1859 (90.51%) | class1= 195 (9.49%) | c1/c0=0.1049)
ValTun | n= 616 | class0= 540 (87.66%) | class1=  76 (12.34%) | c1/c0=0.1407)
ValSel | n= 616 | class0= 530 (86.04%) | class1=  86 (13.96%) | c1/c0=0.1623)
 Test | n= 822 | class0= 730 (88.81%) | class1=  92 (11.19%) | c1/c0=0.1260)
Reward class ratio (original train): 0.1049

Class Ratio Summary (After ADASYN)
Train | n=3796 | class0=1859 (48.97%) | class1=1937 (51.03%) | c1/c0=1.0420)
Data Split Information
Total samples: 4108
Train set (after ADASYN): 3796 samples
  - Class 0: 1859, Class 1: 1937
Valid(Tune) set: 616 samples
  - Class 0: 540, Class 1: 76
Valid(Select) set: 616 samples
  - Class 0: 530, Class 1: 86
Test set: 822 samples (20.0%)
  - Class 0: 730, Class 1: 92

Class ratio (train set after ADASYN): 1.0420
Class ratio used for reward (fixed original train): 0.1049

HYPERPARAMETER GRID SEARCH (Total: 16 combinations)

[1/16] γ=0.90, lr=1e-04, ε=0.990, bs=32


KeyboardInterrupt: 